In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

In [4]:
!pip install bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.8 MB/s eta 0:00:00:00:0100:01


In [28]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [29]:
%%writefile onegpu_baseline.py

import torch
import time
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader
import bitsandbytes as bnb

# ── Config (đồng bộ với Bước 2) ───────────────────────────────
MODEL_NAME  = "gpt2-xl"
SEQ_LEN     = 256
BATCH_SIZE  = 3
GRAD_ACCUM  = 2
MAX_STEPS   = 100
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE       = torch.bfloat16

# ── Helper ─────────────────────────────────────────────────────
def log_memory(tag=""):
    alloc  = torch.cuda.memory_allocated()    / 1024**3
    resv   = torch.cuda.memory_reserved()     / 1024**3
    peak   = torch.cuda.max_memory_allocated() / 1024**3
    print(f"[MEM]{' '+tag+' ' if tag else ' '}"
          f"allocated={alloc:.2f}GB  reserved={resv:.2f}GB  peak={peak:.2f}GB")

# ── 1. Load tokenizer & dataset ────────────────────────────────
print("=" * 60)
print("Bước 1 – OOM Demo: GPT-2 XL | 1 GPU | bf16 | NO Gradient Checkpointing")
print("=" * 60)

print("\n[1/4] Loading tokenizer & dataset ...")
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

dataset   = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=SEQ_LEN,
        padding="max_length",
    )

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])
tokenized = tokenized.filter(lambda x: x["input_ids"].sum() > 0)
dataloader = DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=True)

# ── 2. Load model in bf16 ──────────────────────────────────────
print(f"\n[2/4] Loading {MODEL_NAME} in bf16 ...")
torch.cuda.reset_peak_memory_stats()
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
model.to(DEVICE)
log_memory("after model load")

# ── 3. Setup optimizer ─────────────────────────────────────────
print("\n[3/4] Setting up AdamW 8-bit optimizer ...")
print("  ⚠️  Gradient Checkpointing: DISABLED")
optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=5e-5)
log_memory("after optimizer init")

# ── 4. Training loop ───────────────────────────────────────────
print(f"\n[4/4] Starting training — expecting OOM ...")
print("-" * 60)

model.train()
optimizer_step = 0
micro_step     = 0
start_time     = time.time()
accum_loss     = 0.0

try:
    for batch in dataloader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = input_ids.clone()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()

        accum_loss += loss.item()
        micro_step += 1

        if micro_step % GRAD_ACCUM == 0:
            optimizer.step()
            optimizer.zero_grad()
            optimizer_step += 1

            if optimizer_step % 10 == 0:
                log_memory(f"step {optimizer_step}")
                print(f"  step={optimizer_step:4d}  loss={accum_loss*GRAD_ACCUM:.4f}")

            accum_loss = 0.0

            if optimizer_step >= MAX_STEPS:
                print(f"\n✅ Hoàn thành {MAX_STEPS} steps mà không OOM.")
                print("   → Activations không đủ lớn để trigger OOM với config này.")
                break

except torch.cuda.OutOfMemoryError as e:
    print("\n" + "=" * 60)
    print("💥 CUDA OUT OF MEMORY — OOM confirmed!")
    print("=" * 60)
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"   Peak VRAM allocated : {peak:.2f} GB")
    print(f"   T4 VRAM available   : 16.00 GB")
    print(f"   Overflow            : {max(peak-16, 0):.2f} GB")
    print(f"   Optimizer steps     : {optimizer_step}")
    print(f"   Micro steps         : {micro_step}")
    print("=" * 60)
    print("\nKết luận: Không có Gradient Checkpointing → activations tích lũy")
    print("          → OOM dù đã dùng bf16 + AdamW8bit.")

finally:
    log_memory("final")
    print(f"\nTotal time: {time.time() - start_time:.1f}s")

Writing onegpu_baseline.py


In [30]:
!python onegpu_baseline.py

Bước 1 – OOM Demo: GPT-2 XL | 1 GPU | bf16 | NO Gradient Checkpointing

[1/4] Loading tokenizer & dataset ...
Filter: 100%|███████████████████| 36718/36718 [00:00<00:00, 59593.36 examples/s]

[2/4] Loading gpt2-xl in bf16 ...
Loading weights: 100%|█| 580/580 [00:01<00:00, 407.58it/s, Materializing param=t
GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[MEM] after model load allocated=2.95GB  reserved=2.96GB  peak=2.95GB

[3/4] Setting up AdamW 8-bit optimizer ...
  ⚠️  Gradient Checkpointing: DISABLED
[MEM] after optimizer init allocated=2.95GB  reserved=2.96GB  peak=2.95GB

[4/4] Starting training — expecting OOM ...
------------------------------------------------------------
`loss_type=None` was set in the config but it is unrecognized. Using the defa

In [31]:
import gc
import torch

# Kill mọi tensor còn sót trên GPU
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.reset_peak_memory_stats()

# Clear Python objects
gc.collect()

# Xoá các biến global còn sót (model, optimizer, dataloader, ...)
_keep = {"gc", "torch", "_keep"}
for _var in list(globals().keys()):
    if not _var.startswith("_") and _var not in _keep:
        del globals()[_var]

gc.collect()

# Verify
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated() / 1024**3
    print(f"✅ VRAM sau khi clear: {alloc:.3f} GB")
else:
    print("✅ No CUDA device found — RAM cleared")

print("✅ Python gc cleared")

✅ VRAM sau khi clear: 0.000 GB
✅ Python gc cleared


In [32]:
%%writefile onegpu_GC.py

import torch
import time
import json
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader
import bitsandbytes as bnb

# ── Config ────────────────────────────────────────────────────
MODEL_NAME  = "gpt2-xl"
SEQ_LEN     = 256
BATCH_SIZE  = 3
GRAD_ACCUM  = 2       # effective batch size = BATCH_SIZE * GRAD_ACCUM = 2
MAX_STEPS   = 100
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE       = torch.bfloat16
LOG_FILE    = "step2_metrics.json"

# ── Helper ─────────────────────────────────────────────────────
def get_memory_stats():
    return {
        "allocated_gb": round(torch.cuda.memory_allocated()    / 1024**3, 3),
        "reserved_gb" : round(torch.cuda.memory_reserved()     / 1024**3, 3),
        "peak_gb"     : round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    }

def log_memory(tag=""):
    s = get_memory_stats()
    print(f"[MEM]{' '+tag+' ' if tag else ' '}"
          f"allocated={s['allocated_gb']:.2f}GB  "
          f"reserved={s['reserved_gb']:.2f}GB  "
          f"peak={s['peak_gb']:.2f}GB")

# ── 1. Load tokenizer & dataset ────────────────────────────────
print("=" * 60)
print("Bước 2 – Baseline: GPT-2 XL | 1 GPU | bf16 | Gradient Checkpointing")
print("=" * 60)

print("\n[1/5] Loading tokenizer & dataset ...")
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

def tokenize(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=SEQ_LEN,
        padding="max_length",
    )

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])
tokenized = tokenized.filter(lambda x: x["input_ids"].sum() > 0)
dataloader = DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=True)

print(f"  Dataset size    : {len(tokenized)} samples")
print(f"  Batch size      : {BATCH_SIZE}  (grad_accum={GRAD_ACCUM} → effective bs={BATCH_SIZE*GRAD_ACCUM})")
print(f"  Seq length      : {SEQ_LEN}")

# ── 2. Load model in bf16 ──────────────────────────────────────
print(f"\n[2/5] Loading {MODEL_NAME} in bf16 ...")
torch.cuda.reset_peak_memory_stats()
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
model.to(DEVICE)
log_memory("after model load")

# ── 3. Enable Gradient Checkpointing ──────────────────────────
print("\n[3/5] Enabling Gradient Checkpointing ...")
model.gradient_checkpointing_enable()
print("  ✅ gradient_checkpointing = True")
print("  ℹ️  Recompute activations on backward → giảm VRAM, tăng compute ~20-30%")
log_memory("after GC enable")

# ── 4. Setup optimizer (no GradScaler needed for bf16) ─────────
print("\n[4/5] Setting up AdamW 8-bit optimizer ...")
print("  ℹ️  bf16 không cần GradScaler — dynamic range đủ rộng")
print("  ℹ️  AdamW8bit: optimizer states ở INT8 → giảm ~4x VRAM so FP32 AdamW")
optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=5e-5)
log_memory("after optimizer init")

# ── 5. Training loop ───────────────────────────────────────────
print(f"\n[5/5] Training for {MAX_STEPS} optimizer steps ...")
print("-" * 60)

model.train()
optimizer_step = 0
micro_step     = 0
all_metrics    = []
start_time     = time.time()
accum_loss     = 0.0
step_start     = time.time()

try:
    for batch in dataloader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = input_ids.clone()

        # bf16 forward — no autocast needed, model already in bf16
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss / GRAD_ACCUM
        loss.backward()

        accum_loss += loss.item()
        micro_step += 1

        if micro_step % GRAD_ACCUM == 0:
            optimizer.step()
            optimizer.zero_grad()

            step_time  = time.time() - step_start
            tokens_sec = (BATCH_SIZE * GRAD_ACCUM * SEQ_LEN) / step_time
            mem        = get_memory_stats()

            optimizer_step += 1
            step_metrics = {
                "step"          : optimizer_step,
                "loss"          : round(accum_loss * GRAD_ACCUM, 4),
                "sec_per_step"  : round(step_time, 3),
                "tokens_per_sec": round(tokens_sec, 1),
                "peak_vram_gb"  : mem["peak_gb"],
            }
            all_metrics.append(step_metrics)

            if optimizer_step % 10 == 0:
                print(f"  step={optimizer_step:4d}  loss={step_metrics['loss']:.4f}  "
                      f"tokens/sec={tokens_sec:7.1f}  "
                      f"sec/step={step_time:.3f}s  "
                      f"peak_vram={mem['peak_gb']:.2f}GB")

            accum_loss = 0.0
            step_start = time.time()

            if optimizer_step >= MAX_STEPS:
                break

except torch.cuda.OutOfMemoryError as e:
    print(f"\n💥 OOM at optimizer_step={optimizer_step}, micro_step={micro_step}")
    print(f"   Error: {e}")
    log_memory("OOM point")

# ── Summary ────────────────────────────────────────────────────
total_time = time.time() - start_time

if all_metrics:
    avg_tokens_sec = sum(m["tokens_per_sec"] for m in all_metrics) / len(all_metrics)
    avg_sec_step   = sum(m["sec_per_step"]   for m in all_metrics) / len(all_metrics)
    final_loss     = all_metrics[-1]["loss"]
    peak_vram      = max(m["peak_vram_gb"]   for m in all_metrics)

    print("\n" + "=" * 60)
    print("📊 SUMMARY — Bước 2 (1 GPU + GC + bf16)")
    print("=" * 60)
    print(f"  Steps completed  : {optimizer_step}")
    print(f"  Avg tokens/sec   : {avg_tokens_sec:.1f}")
    print(f"  Avg sec/step     : {avg_sec_step:.3f}s")
    print(f"  Peak VRAM        : {peak_vram:.2f} GB")
    print(f"  Final loss       : {final_loss:.4f}")
    print(f"  Total time       : {total_time:.1f}s")
    print("=" * 60)

    summary = {
        "method"          : "1GPU_GC_bf16_adamw8bit",
        "config"          : {
            "dtype"        : "bfloat16",
            "seq_len"      : SEQ_LEN,
            "batch_size"   : BATCH_SIZE,
            "grad_accum"   : GRAD_ACCUM,
            "effective_bs" : BATCH_SIZE * GRAD_ACCUM,
        },
        "steps"           : optimizer_step,
        "avg_tokens_sec"  : round(avg_tokens_sec, 1),
        "avg_sec_per_step": round(avg_sec_step, 3),
        "peak_vram_gb"    : peak_vram,
        "final_loss"      : final_loss,
        "total_time_sec"  : round(total_time, 1),
        "per_step_metrics": all_metrics,
    }
    with open(LOG_FILE, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\n✅ Metrics saved to {LOG_FILE}")

Writing onegpu_GC.py


In [33]:
!python onegpu_GC.py

Bước 2 – Baseline: GPT-2 XL | 1 GPU | bf16 | Gradient Checkpointing

[1/5] Loading tokenizer & dataset ...
Filter: 100%|███████████████████| 36718/36718 [00:00<00:00, 58624.44 examples/s]
  Dataset size    : 36718 samples
  Batch size      : 3  (grad_accum=2 → effective bs=6)
  Seq length      : 256

[2/5] Loading gpt2-xl in bf16 ...
Loading weights: 100%|█| 580/580 [00:01<00:00, 406.04it/s, Materializing param=t
GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[MEM] after model load allocated=2.95GB  reserved=2.96GB  peak=2.95GB

[3/5] Enabling Gradient Checkpointing ...
  ✅ gradient_checkpointing = True
  ℹ️  Recompute activations on backward → giảm VRAM, tăng compute ~20-30%
[MEM] after GC enable allocated=2.95GB  reserved=2.96GB  peak=2.95GB

[4/5] Set

In [34]:
%%writefile ddp_v2.py

"""
Bước 3: GPT-2 XL Pipeline Parallelism trên 2 GPU
- torch.distributed.pipelining, manual stage split
- bf16, AdamW8bit, BATCH_SIZE=16, SEQ_LEN=256
- Thử chunks = 2, 4, 8, 16
"""

import os, time, json
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.distributed.pipelining import PipelineStage, ScheduleGPipe
from torch.distributed.pipelining.microbatch import TensorChunkSpec
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader, DistributedSampler
import bitsandbytes as bnb
from torch.utils.checkpoint import checkpoint

# ── Config ─────────────────────────────────────────────────────
MODEL_NAME  = "gpt2-xl"
SEQ_LEN     = 256
BATCH_SIZE  = 16
GRAD_ACCUM  = 2
MAX_STEPS   = 100
DTYPE       = torch.bfloat16
LOG_FILE    = "step3_metrics.json"
CHUNKS_LIST = [2, 4, 8, 16]
SPLIT_LAYER = 24

# ── Stage 0 ────────────────────────────────────────────────────
class Stage0(nn.Module):
    def __init__(self, model):
        super().__init__()
        t = model.transformer
        self.wte    = t.wte
        self.wpe    = t.wpe
        self.drop   = t.drop
        self.blocks = nn.ModuleList(t.h[:SPLIT_LAYER])

    def forward(self, input_ids, attention_mask):
        B, T  = input_ids.shape
        pos   = torch.arange(T, device=input_ids.device)
        hidden = self.drop(self.wte(input_ids) + self.wpe(pos))

        causal = torch.tril(torch.ones(T, T, device=hidden.device, dtype=torch.bool))[None, None]
        pad    = attention_mask[:, None, None, :].bool()
        mask   = causal & pad

        for block in self.blocks:
            hidden = checkpoint(block, hidden, use_reentrant=False,
                                attention_mask=mask)[0]
        # Trả về hidden + mask để Stage1 dùng (mask cần để backprop qua cả 2 stage)
        return hidden, mask.to(dtype=hidden.dtype)

# ── Stage 1 ────────────────────────────────────────────────────
class Stage1(nn.Module):
    def __init__(self, model):
        super().__init__()
        t = model.transformer
        self.blocks  = nn.ModuleList(t.h[SPLIT_LAYER:])
        self.ln_f    = t.ln_f
        self.lm_head = model.lm_head

    def forward(self, hidden, mask):
        # Dummy op để giữ gradient flow qua mask
        hidden = hidden + mask.sum() * 0.0
        bool_mask = mask.bool()
        for block in self.blocks:
            hidden = checkpoint(block, hidden, use_reentrant=False,
                                attention_mask=bool_mask)[0]
        return self.lm_head(self.ln_f(hidden))

# ── Loss ───────────────────────────────────────────────────────
def compute_loss(logits, labels):
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    return nn.CrossEntropyLoss()(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
    )

# ── Helpers ────────────────────────────────────────────────────
def get_memory_stats(device):
    return {
        "allocated_gb": round(torch.cuda.memory_allocated(device)    / 1024**3, 3),
        "peak_gb"     : round(torch.cuda.max_memory_allocated(device) / 1024**3, 3),
    }

def get_dataloader(tokenizer, rank, world_size):
    # 1. Cho GPU 0 tải và xử lý data trước
    if rank == 0:
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        def tokenize(examples):
            return tokenizer(examples["text"], truncation=True, max_length=SEQ_LEN, padding="max_length")
        tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
        tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])
        tokenized = tokenized.filter(lambda x: x["input_ids"].sum() > 0)
        
    # 2. Ép TẤT CẢ GPU gặp nhau ở đây (GPU 1 sẽ đợi GPU 0 làm xong việc trên)
    dist.barrier()

    # 3. Bây giờ GPU 1 mới vào việc (tốc độ sẽ tính bằng mili-giây vì đọc thẳng từ cache)
    if rank != 0:
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        def tokenize(examples):
            return tokenizer(examples["text"], truncation=True, max_length=SEQ_LEN, padding="max_length")
        tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
        tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])
        tokenized = tokenized.filter(lambda x: x["input_ids"].sum() > 0)

    # 4. Gặp nhau lần nữa cho chắc cú trước khi chia data
    dist.barrier()

    sampler = DistributedSampler(tokenized, num_replicas=world_size, rank=rank, shuffle=True)
    return DataLoader(tokenized, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True)

# ── Worker ─────────────────────────────────────────────────────
def train_worker(rank, world_size, chunks, result_queue):
    dist.init_process_group(backend="nccl", init_method="env://",
                            world_size=world_size, rank=rank)
    torch.cuda.set_device(rank)
    torch.cuda.reset_peak_memory_stats(rank)
    device = torch.device(f"cuda:{rank}")

    if rank == 0:
        print(f"\n{'='*60}\nPipeline | chunks={chunks} | 2 GPU\n{'='*60}")

    tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    full_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
    full_model.eval()

    stage_mod = Stage0(full_model).to(dtype=DTYPE, device=device) if rank == 0 \
                else Stage1(full_model).to(dtype=DTYPE, device=device)
    del full_model
    torch.cuda.empty_cache()

    print(f"[GPU{rank}] loaded — {get_memory_stats(rank)['allocated_gb']:.2f}GB")

    optimizer = bnb.optim.AdamW8bit(stage_mod.parameters(), lr=5e-5)

    # Build stage & schedule (1 lần duy nhất, không tạo lại trong loop)
    stage = PipelineStage(stage_mod, stage_index=rank,
                          num_stages=world_size, device=device)

    # Stage0 output: (hidden, mask) → 2 tensors → TensorChunkSpec cho cả 2
    # Stage1 nhận (hidden, mask) → labels truyền qua target
    schedule = ScheduleGPipe(
        stage,
        n_microbatches=chunks,
        loss_fn=compute_loss,
        args_chunk_spec=(TensorChunkSpec(0), TensorChunkSpec(0)),
    )

    dataloader = get_dataloader(tokenizer, rank, world_size)
    stage_mod.train()

    optimizer_step = 0
    accum_step     = 0        # đếm số lần schedule.step() để gộp GRAD_ACCUM
    all_metrics    = []
    total_start    = time.time()
    accum_loss     = 0.0

    try:
        for batch in dataloader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = input_ids.clone()

            # ── Đo thời gian bắt đầu từ ĐÂY ──────────────────
            step_start = time.time()

            if rank == 0:
                # Stage0: feed input, nhận gradient từ Stage1
                schedule.step(input_ids, attention_mask)
            else:
                # Stage1: tính loss, backward tự động bởi schedule
                losses = []
                schedule.step(target=labels, losses=losses)
                if losses:
                    accum_loss += sum(l.item() for l in losses) / len(losses)

            accum_step += 1

            # Optimizer step sau GRAD_ACCUM lần schedule.step()
            if accum_step % GRAD_ACCUM == 0:
                optimizer.step()
                optimizer.zero_grad()
                dist.barrier()   # sync 2 GPU trước khi đo thời gian

                step_time = time.time() - step_start
                optimizer_step += 1

                if rank == world_size - 1:
                    # tokens = BATCH_SIZE * GRAD_ACCUM * SEQ_LEN (pipeline ≠ data parallel)
                    tokens_sec = (BATCH_SIZE * GRAD_ACCUM * SEQ_LEN) / step_time
                    mem        = get_memory_stats(rank)
                    p          = world_size
                    bubble     = (p - 1) / (chunks + p - 1)

                    step_metrics = {
                        "step"          : optimizer_step,
                        "loss"          : round(accum_loss, 4),
                        "sec_per_step"  : round(step_time, 3),
                        "tokens_per_sec": round(tokens_sec, 1),
                        "peak_vram_gb"  : mem["peak_gb"],
                        "chunks"        : chunks,
                        "bubble_ratio"  : round(bubble, 4),
                    }
                    all_metrics.append(step_metrics)

                    if optimizer_step % 10 == 0:
                        print(f"  step={optimizer_step:4d}  loss={accum_loss:.4f}  "
                              f"tok/s={tokens_sec:8.1f}  "
                              f"sec/step={step_time:.3f}s  "
                              f"bubble={bubble:.2%}")

                accum_loss = 0.0

                if optimizer_step >= MAX_STEPS:
                    break

    except torch.cuda.OutOfMemoryError as e:
        print(f"[GPU{rank}] 💥 OOM: {e}")

    if rank == world_size - 1 and all_metrics:
        total_time     = time.time() - total_start
        avg_tokens_sec = sum(m["tokens_per_sec"] for m in all_metrics) / len(all_metrics)
        avg_sec_step   = sum(m["sec_per_step"]   for m in all_metrics) / len(all_metrics)
        bubble_ratio   = round((world_size - 1) / (chunks + world_size - 1), 4)

        summary = {
            "method"          : "2GPU_pytorch_pipeline",
            "chunks"          : chunks,
            "bubble_ratio"    : bubble_ratio,
            "config"          : {"dtype": "bfloat16", "seq_len": SEQ_LEN,
                                 "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
                                 "effective_bs": BATCH_SIZE * GRAD_ACCUM,
                                 "split_layer": SPLIT_LAYER},
            "steps"           : optimizer_step,
            "avg_tokens_sec"  : round(avg_tokens_sec, 1),
            "avg_sec_per_step": round(avg_sec_step, 3),
            "peak_vram_gb"    : max(m["peak_vram_gb"] for m in all_metrics),
            "final_loss"      : all_metrics[-1]["loss"],
            "total_time_sec"  : round(total_time, 1),
            "per_step_metrics": all_metrics,
        }
        result_queue.put(summary)
        print(f"\n{'='*60}")
        print(f"📊 chunks={chunks} | tok/s={avg_tokens_sec:.1f} | "
              f"bubble={bubble_ratio:.2%} | peak_vram={summary['peak_vram_gb']:.2f}GB")
        print(f"{'='*60}")

    dist.destroy_process_group()

# ── Main ───────────────────────────────────────────────────────
if __name__ == "__main__":
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"

    world_size  = 2
    all_results = []
    ctx         = mp.get_context("spawn")

    for chunks in CHUNKS_LIST:
        print(f"\n{'#'*60}\n# Thử nghiệm chunks = {chunks}\n{'#'*60}")
        result_queue = ctx.Queue()
        mp.spawn(train_worker, args=(world_size, chunks, result_queue),
                 nprocs=world_size, join=True)
        if not result_queue.empty():
            all_results.append(result_queue.get())
        torch.cuda.empty_cache()
        time.sleep(2)

    with open(LOG_FILE, "w") as f:
        json.dump(all_results, f, indent=2)

    print(f"\n{'='*60}\n✅ Saved to {LOG_FILE}")
    print(f"\n{'chunks':>8} | {'tok/s':>12} | {'bubble':>8} | {'sec/step':>10} | {'loss':>8}")
    print("-" * 58)
    for r in all_results:
        print(f"{r['chunks']:>8} | {r['avg_tokens_sec']:>12.1f} | "
              f"{r['bubble_ratio']:>8.2%} | {r['avg_sec_per_step']:>10.3f}s | "
              f"{r['final_loss']:>8.4f}")

Writing ddp_v2.py


In [8]:
!python3 ddp_v2.py


############################################################
# Thử nghiệm chunks = 2
############################################################

Pipeline | chunks=2 | 2 GPU
Loading weights: 100%|█| 580/580 [00:02<00:00, 264.35it/s, Materializing param=t
Loading weights: 100%|█| 580/580 [00:02<00:00, 248.60it/s, Materializing param=t
GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[GPU0] loaded — 1.58GB
[GPU1] loaded — 1.58GB
Filter: 100%|███████████████████| 36718/36718 [00:

In [8]:
!pip install -q datasets bitsandbytes deepspeed accelerate

In [25]:
%%writefile dp.py

import os, time, json
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.multiprocessing as mp
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader, DistributedSampler
import bitsandbytes as bnb
from torch.utils.checkpoint import checkpoint
import deepspeed

deepspeed.utils.nvtx._range_push = lambda *args, **kwargs: None
deepspeed.utils.nvtx._range_pop  = lambda *args, **kwargs: None
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Config ─────────────────────────────────────────────────────
MODEL_NAME  = "gpt2-xl"
SEQ_LEN     = 256
EFFECTIVE_BS = 16  # Tương đương BATCH_SIZE(16) * GRAD_ACCUM(2) ở bản cũ
MAX_STEPS   = 100
DTYPE       = torch.bfloat16
LOG_FILE    = "step3_deepspeed_metrics.json"
CHUNKS_LIST = [ 4, 8, 16]   # DeepSpeed gọi đây là gradient_accumulation_steps
SPLIT_LAYER = 24              

# ── Stage modules ──────────────────────────────────────────────

class Stage0(nn.Module):
    """Embedding + transformer blocks 0..SPLIT_LAYER-1"""
    def __init__(self, model):
        super().__init__()
        t = model.transformer
        self.wte    = t.wte
        self.wpe    = t.wpe
        self.drop   = t.drop
        self.blocks = nn.ModuleList(t.h[:SPLIT_LAYER])

    def forward(self, inputs):
        input_ids, attention_mask = inputs
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device)
        hidden = self.drop(self.wte(input_ids) + self.wpe(pos))
        
        causal_mask = torch.tril(torch.ones((T, T), device=input_ids.device, dtype=torch.bool))
        causal_mask = causal_mask[None, None, :, :] 
        pad_mask = attention_mask[:, None, None, :].bool() 
        extended_attn_mask = causal_mask & pad_mask  
        
        for block in self.blocks:
            hidden = checkpoint(block, hidden, use_reentrant=False, attention_mask=extended_attn_mask)[0]
            
        # ✅ SỬA Ở ĐÂY: Trả về trực tiếp mask dạng bool, không ép kiểu sang bfloat16 nữa
        return (hidden, extended_attn_mask)

class Stage1(nn.Module):
    """Transformer blocks SPLIT_LAYER..47 + ln_f + lm_head"""
    def __init__(self, model):
        super().__init__()
        t = model.transformer
        self.blocks  = nn.ModuleList(t.h[SPLIT_LAYER:])
        self.ln_f    = t.ln_f
        self.lm_head = model.lm_head

    def forward(self, inputs):
        # ✅ SỬA Ở ĐÂY: Nhận trực tiếp mask dạng bool từ Stage 0
        hidden, extended_attn_mask = inputs
        
        # (Đã xóa dòng dummy gradient và ép kiểu ngược)
        
        for block in self.blocks:
            # Truyền thẳng mask vào block
            hidden = checkpoint(block, hidden, use_reentrant=False, attention_mask=extended_attn_mask)[0]
            
        hidden = self.ln_f(hidden)
        logits = self.lm_head(hidden) 
        return logits


# ── Loss function ──────────────────────────────────────────────

def compute_loss(logits, labels):
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    return nn.CrossEntropyLoss()(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1)
    )

# ── Dataloader Wrapper cho DeepSpeed ───────────────────────────

class DeepSpeedPipelineIterator:
    """DeepSpeed Pipeline yêu cầu Iterator trả về đúng chuẩn ((inputs), labels)"""
    def __init__(self, dataloader):
        self.dataloader = dataloader
        self.iter = iter(self.dataloader)

    def __iter__(self):
        return self

    def __next__(self):
        try:
            batch = next(self.iter)
        except StopIteration:
            self.iter = iter(self.dataloader)
            batch = next(self.iter)
            
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = input_ids.clone()
        
        # Return format: ( (inputs_for_stage0), labels_for_loss )
        return ((input_ids, attention_mask), labels)

# ── Helpers ────────────────────────────────────────────────────

def get_memory_stats(device):
    return {
        "allocated_gb": round(torch.cuda.memory_allocated(device)    / 1024**3, 3),
        "peak_gb"     : round(torch.cuda.max_memory_allocated(device) / 1024**3, 3),
    }

def get_dataloader(tokenizer, rank, world_size, micro_batch_size):
    if rank == 0:
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        def tokenize(examples):
            return tokenizer(examples["text"], truncation=True, max_length=SEQ_LEN, padding="max_length")
        tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
        tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])
        tokenized = tokenized.filter(lambda x: x["input_ids"].sum() > 0)
        
    dist.barrier()

    if rank != 0:
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        def tokenize(examples):
            return tokenizer(examples["text"], truncation=True, max_length=SEQ_LEN, padding="max_length")
        tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
        tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])
        tokenized = tokenized.filter(lambda x: x["input_ids"].sum() > 0)

    dist.barrier()
    sampler = DistributedSampler(tokenized, num_replicas=world_size, rank=rank, shuffle=True)
    return DataLoader(tokenized, batch_size=micro_batch_size, sampler=sampler, drop_last=True)


# ── Training worker ────────────────────────────────────────────

def train_worker(rank, world_size, chunks, result_queue):
    # Setup biến môi trường cho DeepSpeed trong mp.spawn
    os.environ["LOCAL_RANK"] = str(rank)
    os.environ["RANK"] = str(rank)
    os.environ["WORLD_SIZE"] = str(world_size)
    
    deepspeed.init_distributed(dist_backend="nccl")
    torch.cuda.set_device(rank)
    torch.cuda.reset_peak_memory_stats(rank)

    if rank == 0:
        print(f"\n{'='*60}\nDeepSpeed Pipeline | chunks={chunks} | 2 GPU\n{'='*60}")

    tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    full_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
    full_model.eval()

    # DeepSpeed PipelineModule nhận một list các module (sẽ tự động cắt đôi cho 2 GPU)
    layers = [
        Stage0(full_model),
        Stage1(full_model)
    ]

    model = deepspeed.PipelineModule(
        layers=layers,
        num_stages=world_size,
        loss_fn=compute_loss,
        partition_method="parameters",
        activation_checkpoint_interval=0  # Tắt checkpoint mặc định vì mình đã checkpoint thủ công
    )

    del full_model
    torch.cuda.empty_cache()

    mem = get_memory_stats(rank)
    print(f"[GPU{rank}] Stage assigned — allocated={mem['allocated_gb']:.2f}GB")

    # Cấu hình tính toán Micro-batch
    micro_batch_size = EFFECTIVE_BS // chunks

    ds_config = {
        "train_batch_size": EFFECTIVE_BS,
        "train_micro_batch_size_per_gpu": micro_batch_size,
        "gradient_accumulation_steps": chunks,
        "bf16": {"enabled": True},
        "zero_allow_untested_optimizer": True,
        # 🚀 BẬT TÍNH NĂNG ZERO TẠI ĐÂY
        "zero_optimization": {
            "stage": 1,                   # Chỉ dùng Stage 1 khi kết hợp với Pipeline
            "reduce_bucket_size": 5e8,    # Tối ưu kích thước gói tin gửi qua NCCL
            "allgather_bucket_size": 5e8
        }
    }

    optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=5e-5)

    # Khởi tạo DeepSpeed Engine
    engine, optimizer, _, _ = deepspeed.initialize(
        model=model,
        optimizer=optimizer,
        config=ds_config
    )

    dataloader = get_dataloader(tokenizer, rank, world_size, micro_batch_size)
    data_iter = iter(DeepSpeedPipelineIterator(dataloader))

    all_metrics = []
    start_time = time.time()

    try:
        for step in range(1, MAX_STEPS + 1):
            step_start = time.time()
            
            # CHỈ 1 DÒNG DUY NHẤT: train_batch tự handle mọi thứ!
            loss = engine.train_batch(data_iter=data_iter)

            if rank == world_size - 1:
                step_time = time.time() - step_start
                tokens_sec = (EFFECTIVE_BS * SEQ_LEN) / step_time
                mem = get_memory_stats(rank)
                bubble = (world_size - 1) / (chunks + world_size - 1)

                step_metrics = {
                    "step"          : step,
                    "loss"          : round(loss.item(), 4),
                    "sec_per_step"  : round(step_time, 3),
                    "tokens_per_sec": round(tokens_sec, 1),
                    "peak_vram_gb"  : mem["peak_gb"],
                    "chunks"        : chunks,
                    "bubble_ratio"  : round(bubble, 4),
                }
                all_metrics.append(step_metrics)

                if step % 10 == 0:
                    print(f"  step={step:4d}  loss={step_metrics['loss']:.4f}  "
                          f"tok/s={tokens_sec:7.1f}  "
                          f"sec/step={step_time:.3f}s  "
                          f"bubble={bubble:.2%}")

    except torch.cuda.OutOfMemoryError as e:
        print(f"\n[GPU{rank}] 💥 OOM: {e}")

    # Summary
    if rank == world_size - 1 and all_metrics:
        total_time = time.time() - start_time
        avg_tokens_sec = sum(m["tokens_per_sec"] for m in all_metrics) / len(all_metrics)
        avg_sec_step = sum(m["sec_per_step"] for m in all_metrics) / len(all_metrics)
        bubble_ratio = round((world_size - 1) / (chunks + world_size - 1), 4)

        summary = {
            "method"          : "2GPU_deepspeed_pipeline",
            "chunks"          : chunks,
            "bubble_ratio"    : bubble_ratio,
            "steps"           : step,
            "avg_tokens_sec"  : round(avg_tokens_sec, 1),
            "avg_sec_per_step": round(avg_sec_step, 3),
            "peak_vram_gb"    : max(m["peak_vram_gb"] for m in all_metrics),
            "final_loss"      : all_metrics[-1]["loss"],
            "total_time_sec"  : round(total_time, 1),
        }
        result_queue.put(summary)

    dist.destroy_process_group()


# ── Main ───────────────────────────────────────────────────────

if __name__ == "__main__":
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"

    world_size  = 2
    all_results = []
    ctx         = mp.get_context("spawn")

    for chunks in CHUNKS_LIST:
        print(f"\n{'#'*60}\n# Thử nghiệm chunks = {chunks}\n{'#'*60}")
        result_queue = ctx.Queue()

        mp.spawn(
            train_worker,
            args=(world_size, chunks, result_queue),
            nprocs=world_size,
            join=True,
        )

        if not result_queue.empty():
            all_results.append(result_queue.get())

        torch.cuda.empty_cache()
        time.sleep(2)

    with open(LOG_FILE, "w") as f:
        json.dump(all_results, f, indent=2)

    print(f"\n{'='*60}")
    print(f"✅ Saved to {LOG_FILE}")
    print(f"\n{'chunks':>8} | {'tok/s':>10} | {'bubble':>8} | {'sec/step':>10}")
    print("-" * 46)
    for r in all_results:
        print(f"{r['chunks']:>8} | {r['avg_tokens_sec']:>10.1f} | "
              f"{r['bubble_ratio']:>8.2%} | {r['avg_sec_per_step']:>10.3f}s")

Overwriting dp.py


In [26]:
!python dp.py


############################################################
# Thử nghiệm chunks = 4
############################################################

DeepSpeed Pipeline | chunks=4 | 2 GPU
Loading weights: 100%|█| 580/580 [00:02<00:00, 246.67it/s, Materializing param=t
Loading weights: 100%|█| 580/580 [00:02<00:00, 242.04it/s, Materializing param=t
GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
SEED_LAYERS=False BASE_SEED=1234 SEED_FN=None
Using topology: {ProcessCoord(pipe=0, da